# 47-strain IDBac knowledgebase evaluation

This focused notebook reproduces supplementary Figure 6 from the frozen combined-query search tables. It compares the contrastive transformer with intensity-agnostic cosine using WGS genus labels. The 16S rRNA comparison and exploratory analyses are omitted.

Source: `data/ML_Queries_Round_1/results.ipynb` in the IDBac workflow checkout. Despite that notebook's location, its active input paths refer to `ML_Queries_Full`. Provenance and deployment checks are recorded in `../PROVENANCE.md`.

Run from this notebook directory after installing `publication/requirements-analysis.txt`. New tables and figures are written to `../regenerated_results/`; frozen inputs and original figures remain available for comparison.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
DATA = ROOT / 'data'
OUTPUT = ROOT / 'regenerated_results'
OUTPUT.mkdir(parents=True, exist_ok=True)
metadata = pd.read_csv(DATA / 'ML_Query_Metadata.xlsx - Metadata sheet.csv')
ml_result = pd.read_csv(DATA / 'fbb91febffc54470b91d3b84e1ab7293-enriched_db_results.tsv', sep='\t')
presence_result = pd.read_csv(DATA / 'fedcb489d6f7441eb738260b4bb352ab-enriched_db_results.tsv', sep='\t')
kb = pd.read_csv(DATA / 'IDBac_KB_Spectra_List.csv')

## Cohort and taxonomy

The source excludes three uncertain query annotations, leaving the manuscript's 47 strains. WGS genus labels include the source's Rothia reassignment. Existing database annotations use the same genus-normalization fallbacks as the source analysis; these are metadata handling, rather than a 16S comparison.

In [ ]:
metadata['Genus'] = metadata['GTDB WGS taxonomy abbreviated'].str.split().str[0]
metadata.loc[metadata['Strain name'] == '01KA9R76ZXSVEW9GXYZGS85PHJ', 'Genus'] = 'Rothia' # Reclassified by WGS
remove_list = ['01KA9R5KF1QKAGN54SDPMNK7SC.mzML', '01KA9R4XXSNR6R8Z38DAYG2YHE.mzML', '01KA9R753Q02A6K9AC8F0RRNRW.mzML']
cohort = metadata.loc[~metadata['Filename'].isin(remove_list)].copy()
roster = pd.read_csv(DATA / 'manuscript_roster.csv')
assert len(cohort) == 47
assert set(cohort['Strain name']) == set(roster['IDBac-KB ID'])

kb.loc[kb['database_id'] == '01K9QYSKG82XYQ8VXEKXX663XN', 'genus'] = 'Pseudarthrobacter'
kb.genus = kb.genus.fillna(kb['16S Taxonomy'].str.split().str[0])
kb_genera = kb.genus.value_counts() - 1
kb_genera['Pseudarthrobacter'] = 1

## Search filtering

Transformer distance cutoff: 0.232. Cosine distance cutoff: 0.65. The original notebook expressed the transformer cutoff as `1 - 0.232`; every frozen transformer hit has distance below 0.19, so applying 0.232 selects the identical input rows.

The original near-zero-distance exclusion (`distance > 1e-4`) is retained. Distance sorting, missing-genus normalization, and cohort exclusions follow source cells 12–16.

In [ ]:
import numpy as np
def normalize_db_genus(df):
    df = df.copy(deep=True)

    # Replace "Unknown" with NaN for easier handling
    df['db_genus'] = df['db_genus'].replace(r'(?i)^\s*unclassified\s*$', np.nan, regex=True)


    missing_genus = df['db_genus'].isna()
    print(f"Missing db_genus for {missing_genus.sum()} rows")
    if missing_genus.any():
        genus_by_database_id = metadata.set_index('Strain name')['Genus']
        df.loc[missing_genus, 'db_genus'] = df.loc[missing_genus, 'database_id'].map(genus_by_database_id)

    # Replace "Unknown" with NaN for easier handling
    df['db_genus'] = df['db_genus'].replace(r'(?i)^\s*unclassified\s*$', np.nan, regex=True)

    still_missing = df['db_genus'].isna() & df['db_taxonomy'].notna()
    if still_missing.any():
        df.loc[still_missing, 'db_genus'] = df.loc[still_missing, 'db_taxonomy'].str.split().str[0]

    print(f"After normalization, missing db_genus for {df['db_genus'].isna().sum()} rows")

    return df


assert ml_result['distance'].max() < 0.232
ml_result = ml_result[ml_result['distance'] <= 0.232]
ml_result = ml_result[ml_result['distance'] > 1e-4].sort_values('distance')
ml_result = normalize_db_genus(ml_result)
presence_result = presence_result[presence_result['distance'] > 1e-4].sort_values('distance')
presence_result = normalize_db_genus(presence_result)
presence_result = presence_result[presence_result['distance'] <= 0.65]
ml_result = ml_result[~ml_result['query_filename'].isin(remove_list)]
presence_result = presence_result[~presence_result['query_filename'].isin(remove_list)]
ml_result_dict = {q: group.to_dict(orient='records') for q, group in ml_result.groupby('query_filename')}
presence_result_dict = {q: group.to_dict(orient='records') for q, group in presence_result.groupby('query_filename')}

## Top-k scoring

For k = 1–5, accuracy is the number of correct genera among the first k hits divided by k. Missing predictions count as incorrect. Standard averaging weights each strain equally; macro averaging first averages within each genus, then weights the 17 genera equally.

In [ ]:
accuracy_results = []
for _, query in cohort.iterrows():
    strain = query['Strain name']
    true_genus = query['Genus']
    ml_hits = ml_result_dict.get(f'{strain}.mzML', [])
    presence_hits = presence_result_dict.get(f'{strain}.mzML', [])
    for k in range(1, 6):
        ml_genera = [hit['db_genus'] for hit in ml_hits[:k]]
        presence_genera = [hit['db_genus'] for hit in presence_hits[:k]]
        accuracy_results.append({
            'query': strain, 'strain_name': strain, 'true_genus': true_genus,
            'true_isin_kb': kb_genera.get(true_genus, 0) > 0,
            'count_others_in_kb': kb_genera.get(true_genus, 0),
            'ml_accuracy': sum(g == true_genus for g in ml_genera) / k,
            'presence_accuracy': sum(g == true_genus for g in presence_genera) / k,
            'ml_has_correct_hit': any(g == true_genus for g in ml_genera),
            'presence_has_correct_hit': any(g == true_genus for g in presence_genera),
            'ml_has_preds': bool(ml_genera), 'presence_has_preds': bool(presence_genera), 'k': k,
        })
accuracy_df = pd.DataFrame(accuracy_results)
accuracy_df.to_csv(OUTPUT / 'per_query_top_k.csv', index=False)

saved_top1 = pd.read_csv(DATA / 'gemini_first_output.csv').set_index('query')
top1 = accuracy_df.loc[accuracy_df['k'] == 1].set_index('query')
for column in ['true_genus', 'ml_accuracy', 'presence_accuracy', 'ml_has_preds', 'presence_has_preds']:
    pd.testing.assert_series_equal(top1[column].sort_index(), saved_top1[column].sort_index(), check_names=False)
assert top1['ml_accuracy'].sum() == 32
assert top1['presence_accuracy'].sum() == 26

In [ ]:
method_names = {'ml_accuracy': 'Contrastive Transformer', 'presence_accuracy': 'Cosine Similarity (Int. Agn.)'}
long = accuracy_df.melt(id_vars=['query', 'true_genus', 'k'], value_vars=list(method_names), var_name='method', value_name='accuracy')
standard = long.groupby(['method', 'k'])['accuracy'].mean().reset_index()
genus_means = long.groupby(['method', 'true_genus', 'k'])['accuracy'].mean().reset_index()
macro = genus_means.groupby(['method', 'k'])['accuracy'].mean().reset_index()
standard.to_csv(OUTPUT / 'standard_top_k.csv', index=False)
macro.to_csv(OUTPUT / 'macro_top_k.csv', index=False)
macro_top1 = macro.loc[macro['k'] == 1].set_index('method')['accuracy']
assert np.isclose(macro_top1['ml_accuracy'], 0.38745098039215686)
assert np.isclose(macro_top1['presence_accuracy'], 0.1988235294117647)
display(standard, macro)

## Supplementary Figure 6

The dashed reference line below preserves the source notebook's historical `thr_max_calc` convention, including its `k - 1` condition. This convention is recorded as a methods caveat in the provenance document; it has no effect on the measured method accuracies. The original saved panels are included in `../figures/`.

In [ ]:
colors = {
    'Cosine Similarity (Int. Agn.)': '#e78ac3',
    'Contrastive Transformer': '#66c2a5',
    'Theoretical Max': "#727272",
}
plt.rcParams.update({'font.size': 18, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial']})
def thr_max_calc(row):
    if row['count_others_in_kb'] == 0:
        return 0.0
    if row['count_others_in_kb'] >= row['k'] - 1:
        return 1.0
    return row['count_others_in_kb'] / row['k']
accuracy_df['query_theoretical_max'] = accuracy_df.apply(thr_max_calc, axis=1)
standard_reference = accuracy_df.groupby('k')['query_theoretical_max'].mean()
macro_reference = accuracy_df.groupby(['true_genus', 'k'])['query_theoretical_max'].mean().groupby('k').mean()
for table, reference, filename, ylabel, reference_label in [
    (standard, standard_reference, 'top_k_accuracy', 'Top-k Accuracy', 'Theoretical Max'),
    (macro, macro_reference, 'macro_top_k_accuracy', 'Macro Top-k Accuracy', 'Macro Theoretical Max'),
]:
    fig, ax = plt.subplots(figsize=(5, 3))
    for method, name in method_names.items():
        values = table.loc[table['method'] == method]
        ax.plot(values['k'], values['accuracy'], marker='o', label=name, color=colors[name])
    ax.plot(reference.index, reference.values, color=colors['Theoretical Max'], linestyle='--', label=reference_label)
    ax.set(xlabel='# Hits Considered (k)', ylabel=ylabel, ylim=(0, reference.max() + 0.1), xticks=range(1, 6))
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    fig.savefig(OUTPUT / f'{filename}.png', dpi=200, bbox_inches='tight')
    fig.savefig(OUTPUT / f'{filename}.svg', bbox_inches='tight')
    plt.show()

Original saved panels, extracted from source cells 23 and 26:

![Standard top-k accuracy](../figures/top_k_accuracy.png)

![Genus-weighted top-k accuracy](../figures/macro_top_k_accuracy.png)